In [19]:
import numpy as np
import pandas as pd

from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

try:
    from pgmpy.models import DiscreteBayesianNetwork
    BayesianModel = DiscreteBayesianNetwork
except ImportError:
    from pgmpy.models import BayesianNetwork
    BayesianModel = BayesianNetwork

In [20]:
data = pd.read_csv(
    r"C:\Users\Dell\OneDrive\Desktop\ir_lab\heart.csv"
)

data = data.replace('?', np.nan)

print("First five records:")
print(data.head())

print("\nDataset shape:", data.shape)
print("\nColumn names:")
print(data.columns.tolist())

First five records:
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   52    1   0       125   212    0        1      168      0      1.0      2   
1   53    1   0       140   203    1        0      155      1      3.1      0   
2   70    1   0       145   174    0        1      125      1      2.6      0   
3   61    1   0       148   203    0        1      161      0      0.0      2   
4   62    0   0       138   294    1        1      106      0      1.9      1   

   ca  thal  target  
0   2     3       0  
1   0     3       0  
2   0     3       0  
3   1     3       0  
4   3     2       0  

Dataset shape: (1025, 14)

Column names:
['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']


In [21]:
data = data.rename(columns={'target': 'heartdisease'})

cols = [
    'age',
    'sex',
    'cp',
    'exang',
    'restecg',
    'chol',
    'heartdisease'
]

data = data[cols]

data = data.apply(pd.to_numeric, errors='coerce')

data = data.dropna()

print("Dataset after removing missing values:")
print(data.head())

Dataset after removing missing values:
   age  sex  cp  exang  restecg  chol  heartdisease
0   52    1   0      0        1   212             0
1   53    1   0      1        0   203             0
2   70    1   0      1        1   174             0
3   61    1   0      0        1   203             0
4   62    0   0      0        1   294             0


In [22]:
data['age'] = pd.cut(
    data['age'],
    bins=[0, 40, 55, 70, 120],
    labels=False
)

data['chol'] = pd.cut(
    data['chol'],
    bins=[0, 200, 240, 1000],
    labels=False
)

data = data.dropna()

data = data.astype(int)

print("Processed Dataset:")
print(data.head())

print("\nDataset shape:", data.shape)

Processed Dataset:
   age  sex  cp  exang  restecg  chol  heartdisease
0    1    1   0      0        1     1             0
1    1    1   0      1        0     1             0
2    2    1   0      1        1     0             0
3    2    1   0      0        1     1             0
4    2    0   0      0        1     2             0

Dataset shape: (1025, 7)


In [23]:
model = BayesianModel([
    ('age', 'heartdisease'),
    ('sex', 'heartdisease'),
    ('exang', 'heartdisease'),
    ('cp', 'heartdisease'),
    ('heartdisease', 'restecg'),
    ('heartdisease', 'chol')
])

print("Bayesian Network Structure:")
print(model.edges())

Bayesian Network Structure:
[('age', 'heartdisease'), ('heartdisease', 'restecg'), ('heartdisease', 'chol'), ('sex', 'heartdisease'), ('exang', 'heartdisease'), ('cp', 'heartdisease')]


In [26]:
from pgmpy.parameter_estimator import DiscreteMLE

estimator = DiscreteMLE()

model.fit(
    data,
    estimator=estimator
)

print("Bayesian Network trained successfully!")

Bayesian Network trained successfully!


In [27]:
infer = VariableElimination(model)

print("Inference initialized successfully!")

Inference initialized successfully!


In [28]:
result1 = infer.query(
    variables=['heartdisease'],
    evidence={'restecg': 1}
)

print("Prediction using Resting ECG:")
print(result1)

Prediction using Resting ECG:
+-----------------+---------------------+
| heartdisease    |   phi(heartdisease) |
+=================+=====================+
| heartdisease(0) |              0.3889 |
+-----------------+---------------------+
| heartdisease(1) |              0.6111 |
+-----------------+---------------------+
